# Перенос обучения и fine-tuning

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Переносить обученное представление на другую задачу и различать заморозку и fine-tuning.

## Повторное использование признаков

Перенос обучения берёт параметры, уже обученные на исходной задаче, и использует их в новой. Сначала можно заморозить извлекатель признаков и учить только новую «голову» — последний преобразователь ответов. Затем fine-tuning разрешает обновлять часть или все перенесённые веса меньшим шагом.

Заморозка requires_grad_(False) запрещает вычисление градиента по параметрам. eval меняет режим некоторых слоёв, но сам по себе не замораживает веса. Эти действия нельзя заменять друг другом.

## Разобранный пример: маленькая исходная задача

Вместо скачивания большой модели обучим представление по точкам x: исходная цель 2x. Извлекатель — линейный слой с двумя выходами, голова — слой 2→1. Затем новая цель 2x+1: сохраним извлекатель, создадим новую голову и обучим её. Оба закона авторские, поэтому опыт показывает механику переноса, а не пользу предобучения на реальных изображениях.

Перед переносом проверяют совместимость входа: форму, масштаб, каналы и нормировку. Замена головы исправляет число выходов, но не делает произвольные входные данные подходящими. Перенос способен ухудшить результат, если исходное представление потеряло нужную новой задаче информацию.

## Проверяем действительно ли веса сохранены

Скопируем веса извлекателя до обучения новой головы, после сравним их точно. Проверка снижения ошибки новой цели подтверждает работу маленького опыта. Для честного сравнения с обучением с нуля нужны независимые данные и одинаковый бюджет, чего этот иллюстративный расчёт сам по себе не доказывает.

В коде оптимизатор новой задачи получает только `head.parameters()`, а `backbone.requires_grad_(False)` дополнительно фиксирует извлекатель. Обычный цикл обучения уже знаком из предыдущих уроков; все модели создаются и обучаются здесь на CPU.

В проверках `all(список_условий)` возвращает True, только если истинны все условия списка. Это позволяет одним утверждением проверить каждый параметр или каждую строку результата. Для пустого списка `all` тоже даёт True; поэтому непустоту данных при необходимости проверяют отдельно.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

Перед переносом проверяем ошибку исходной задачи. После заморозки `backbone.zero_grad(set_to_none=True)` удаляет градиенты, оставшиеся от её обучения; сама заморозка их не стирает. Затем проверяем и неизменность весов извлекателя, и отсутствие новых градиентов у него. Итоговую ошибку целевой задачи считаем в `eval` и `no_grad`.

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(17)
x = torch.linspace(-1, 1, 20).reshape(-1, 1)
backbone = nn.Linear(1, 2)
source_head = nn.Linear(2, 1)
source = nn.Sequential(backbone, source_head)
optimizer = torch.optim.SGD(source.parameters(), lr=0.05)
for _ in range(120):
    optimizer.zero_grad(set_to_none=True)
    loss = ((source(x) - 2*x)**2).mean()
    loss.backward()
    optimizer.step()
source.eval()
with torch.no_grad():
    source_error = ((source(x) - 2*x)**2).mean().item()
assert source_error < 0.01
backbone.requires_grad_(False)
backbone.zero_grad(set_to_none=True)
saved = [p.detach().clone() for p in backbone.parameters()]
head = nn.Linear(2, 1)
target = nn.Sequential(backbone, head)
optimizer = torch.optim.SGD(head.parameters(), lr=0.05)
y = 2*x + 1
initial = ((target(x) - y)**2).mean().item()
for _ in range(120):
    optimizer.zero_grad(set_to_none=True)
    loss = ((target(x) - y)**2).mean()
    loss.backward()
    optimizer.step()
assert all([torch.equal(a, b) for a, b in zip(saved, backbone.parameters())])
assert all([p.grad is None for p in backbone.parameters()])
target.eval()
with torch.no_grad():
    final = ((target(x)-y)**2).mean().item()
assert final < initial * 0.1
print("Исходная и целевая задачи обучены здесь; извлекатель сохранён")


## Типичные ошибки

Не называйте созданную здесь сеть большой предобученной моделью. Сверяйте входную подготовку и состав параметров оптимизатора.

## Задачи

### 1. Базовая

Что называют головой?

<details>
<summary>Подсказка</summary>

Это часть, выдающая ответ конкретной задачи.

</details>



### 2. Базовая

Чем fine-tuning отличается от заморозки?

<details>
<summary>Подсказка</summary>

Могут ли меняться перенесённые веса?

</details>



### 3. Прикладная

Исходная голова 8→10, новая задача с тремя классами. Какой выход нужен?

<details>
<summary>Подсказка</summary>

Число счётов равно числу классов.

</details>



### 4. Прикладная

Как проверить, что замороженные веса не изменились?

<details>
<summary>Подсказка</summary>

Нужна независимая копия до опыта.

</details>



### 5. Олимпиадная

Гарантирует ли перенос преимущество перед обучением с нуля?

<details>
<summary>Подсказка</summary>

Исходная задача может быть неподходящей.

</details>



### 6. Олимпиадная

Почему eval не заменяет requires_grad=False?

<details>
<summary>Подсказка</summary>

Режим и отслеживание производных независимы.

</details>



## Итоги и самопроверка

Опишите два режима переноса и способ проверить, что веса извлекателя `backbone` не изменились. Какой эксперимент понадобился бы для доказательства полезности переноса?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Последние слои, преобразующие представление в нужный выход.

### Задача 2

При fine-tuning выбранные перенесённые параметры тоже обновляются.

### Задача 3

Новая голова 8→3 при сохранённой размерности представления.

### Задача 4

Сравнить все параметры с сохранёнными значениями после обучения новой головы.

### Задача 5

Нет. Возможен отрицательный перенос; нужно сравнение при одинаковом протоколе.

### Задача 6

Параметры в eval могут получать градиенты и обновляться оптимизатором.